In [5]:
knowledge_graph = {
    "checkout_failures": [
        {"node": "checkout_api", "relationship": "affects"},
        {"node": "payment_errors", "relationship": "correlates_with"},
    ],

    "checkout_api": [
        {"node": "payment_service", "relationship": "depends_on"},
        {"node": "inventory_service", "relationship": "depends_on"},
    ],

    "payment_errors": [
        {"node": "payment_service", "relationship": "generated_by"},
    ],

    "payment_service": [
        {"node": "database_latency", "relationship": "experiencing"},
        {"node": "expired_certificate", "relationship": "possible_cause"},
    ],

    "inventory_service": [
        {"node": "inventory_database", "relationship": "depends_on"},
    ],

    "database_latency": [
        {"node": "connection_pool_exhaustion", "relationship": "caused_by"},
    ],

    "inventory_database": [
        {"node": "connection_pool_exhaustion", "relationship": "experiencing"},
    ],

    "expired_certificate": [],
    "connection_pool_exhaustion": [],
}

In [6]:
from collections import deque

In [28]:
def find_evidence_path(graph, start, target):
    queue = deque()
    visited = set()
    queue.append((start, [start])) #identify starting node based on input to function
    visited.add(start) # add start node to visited
    while queue:
        current_node, evidence_path = queue.popleft()

        if current_node == target:
            return evidence_path
        for edge in graph.get(current_node, []):
            neighbor_node = edge["node"]
            relationship = edge["relationship"]
            if neighbor_node not in visited:
                visited.add(neighbor_node)
                new_relationship = ({"from": current_node,"relationship": relationship, "to": neighbor_node})
                new_evidence_path = evidence_path + [new_relationship]
                print(
                    f'Discovered {neighbor_node} through '
                    f'{current_node}-->{relationship}-->{neighbor_node}'
                )
                queue.append((neighbor_node, new_evidence_path))
                print(queue)

    return None




In [29]:
find_evidence_path(knowledge_graph, start="checkout_failures",
    target="connection_pool_exhaustion",
)

Discovered checkout_api through checkout_failures-->affects-->checkout_api
deque([('checkout_api', ['checkout_failures', {'from': 'checkout_failures', 'relationship': 'affects', 'to': 'checkout_api'}])])
Discovered payment_errors through checkout_failures-->correlates_with-->payment_errors
deque([('checkout_api', ['checkout_failures', {'from': 'checkout_failures', 'relationship': 'affects', 'to': 'checkout_api'}]), ('payment_errors', ['checkout_failures', {'from': 'checkout_failures', 'relationship': 'correlates_with', 'to': 'payment_errors'}])])
Discovered payment_service through checkout_api-->depends_on-->payment_service
deque([('payment_errors', ['checkout_failures', {'from': 'checkout_failures', 'relationship': 'correlates_with', 'to': 'payment_errors'}]), ('payment_service', ['checkout_failures', {'from': 'checkout_failures', 'relationship': 'affects', 'to': 'checkout_api'}, {'from': 'checkout_api', 'relationship': 'depends_on', 'to': 'payment_service'}])])
Discovered inventory_s

['checkout_failures',
 {'from': 'checkout_failures',
  'relationship': 'affects',
  'to': 'checkout_api'},
 {'from': 'checkout_api',
  'relationship': 'depends_on',
  'to': 'payment_service'},
 {'from': 'payment_service',
  'relationship': 'experiencing',
  'to': 'database_latency'},
 {'from': 'database_latency',
  'relationship': 'caused_by',
  'to': 'connection_pool_exhaustion'}]